# Stage 9 Explainer Notebook

This notebook is the Stage 9 system-engineering dashboard.
Use it to run scripts, inspect middle artifacts, simulate production controls, and verify release evidence.


## Prerequisites
- Run this notebook from `red-book/src/stage-9` if possible.
- Install dependencies first:
  - `pip install -r requirements.txt`
  - optional: `pip install -r requirements-optional.txt`
- Execute cells in order.


In [ ]:
from __future__ import annotations

import csv
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    import torch
except Exception:
    torch = None

STAGE = 9
CWD = Path.cwd()

if (CWD / f"run_all_stage{STAGE}.ps1").exists():
    STAGE_DIR = CWD
else:
    STAGE_DIR = Path(r"C:/Users/luixj/AI/red-book/src/stage-9").resolve()

RESULTS_DIR = STAGE_DIR / "results"
STAGE_RESULTS_DIR = RESULTS_DIR / f"stage{STAGE}"
STAGE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Stage:", STAGE)
print("Stage dir:", STAGE_DIR)
print("Results dir:", RESULTS_DIR)
print("Canonical stage results dir:", STAGE_RESULTS_DIR)


def run_cmd(cmd, cwd=STAGE_DIR):
    if isinstance(cmd, str):
        cmd_list = cmd.split()
    else:
        cmd_list = cmd
    print("$", " ".join(map(str, cmd_list)))
    proc = subprocess.run(
        cmd_list,
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print("[stderr]\n" + proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit={proc.returncode}): {cmd_list}")
    return proc


def snapshot_results():
    if not RESULTS_DIR.exists():
        return {}
    snap = {}
    for p in RESULTS_DIR.rglob('*'):
        if p.is_file():
            snap[str(p.relative_to(RESULTS_DIR))] = p.stat().st_mtime
    return snap


def diff_results(before, after):
    new_files = sorted([name for name in after if name not in before])
    changed_files = sorted([
        name for name in after
        if name in before and after[name] != before[name]
    ])
    return new_files, changed_files


def run_script(script_name: str):
    before = snapshot_results()
    script_path = STAGE_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Missing script: {script_path}")
    run_cmd([sys.executable, script_name])
    after = snapshot_results()
    new_files, changed_files = diff_results(before, after)
    print("new files:", new_files)
    print("changed files:", changed_files)


def list_results(limit=120):
    if not RESULTS_DIR.exists():
        print("results directory does not exist yet")
        return
    files = sorted([p for p in RESULTS_DIR.rglob('*') if p.is_file()])
    print(f"total files: {len(files)}")
    for p in files[:limit]:
        rel = p.relative_to(RESULTS_DIR)
        print(f"- {rel} ({p.stat().st_size} bytes)")


def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
    return rows


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def preview_csv_path(path: Path, rows: int = 6):
    if not path.exists():
        print("missing:", path)
        return
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            print(row)
            if i + 1 >= rows:
                break


def preview_json_path(path: Path, rows: int = 5):
    if not path.exists():
        print("missing:", path)
        return
    if path.suffix.lower() == '.jsonl':
        with path.open('r', encoding='utf-8', errors='replace') as f:
            for i, line in enumerate(f):
                print(line.rstrip())
                if i + 1 >= rows:
                    break
    else:
        print(path.read_text(encoding='utf-8', errors='replace')[:1800])



## 1) Preflight Checks


In [ ]:
run_cmd([sys.executable, '--version'])

runner = STAGE_DIR / 'run_all_stage9.ps1'
verifier = STAGE_DIR / 'verify_stage9_outputs.ps1'
print('runner exists:', runner.exists(), runner)
print('verifier exists:', verifier.exists(), verifier)
print('lab scripts:', sorted(p.name for p in STAGE_DIR.glob('lab*.py')))
print('artifact map exists:', (STAGE_DIR / 'artifact_name_map.md').exists())

if torch is not None:
    print('torch version:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
else:
    print('torch not importable in this environment')



## 1.1) GPU Telemetry Baseline (RTX 5090 focus)


In [ ]:
if torch is None:
    print('torch not available; cannot run GPU telemetry baseline')
else:
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('Device:', torch.cuda.get_device_name(0))
        try:
            torch.cuda.reset_peak_memory_stats(0)
            x = torch.randn((2048, 2048), device='cuda')
            y = torch.randn((2048, 2048), device='cuda')
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            z = x @ y
            torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            peak_mb = torch.cuda.max_memory_allocated(0) / (1024**2)
            print(f'GPU matmul time: {dt:.4f}s | peak VRAM: {peak_mb:.2f} MB')
            print(torch.cuda.memory_summary(device=0, abbreviated=True)[:1500])
        except Exception as e:
            print('GPU telemetry baseline failed:', e)
    else:
        print('CUDA unavailable: CPU fallback mode')



## 2) Run Intermediate Topic Scripts (Trace Middle Artifacts)


In [ ]:
RUN_ADVANCED = False

topic_scripts = [
  'topic00_system_flow_intermediate.py',
  'topic01_qdrant_retrieval_intermediate.py',
  'topic02_vllm_serving_intermediate.py',
  'topic03_queue_backpressure_intermediate.py',
  'topic04_pytorch_amp_intermediate.py',
  'topic05_metrics_tracing_intermediate.py',
  'topic06_retry_timeout_intermediate.py',
  'topic07_k8s_deploy_intermediate.py',
  'topic08_tradeoff_analysis_intermediate.py',
]

advanced_scripts = [
  'topic00c_system_flow_advanced.py',
  'topic01c_qdrant_failure_diagnostics_advanced.py',
  'topic02c_ray_serve_orchestration_advanced.py',
  'topic03c_autoscaling_policy_advanced.py',
  'topic04c_cuda_oom_recovery_advanced.py',
  'topic05c_incident_triage_advanced.py',
  'topic06c_sla_slo_error_budget_advanced.py',
  'topic07c_canary_rollback_advanced.py',
  'topic08c_architecture_review_board_advanced.py',
]

if RUN_ADVANCED:
    topic_scripts = topic_scripts + advanced_scripts

print('topic scripts:', topic_scripts)
for script in topic_scripts:
    print('\n=== running', script, '===')
    run_script(script)



## 3) Run Stage Labs (Full Workflow Artifacts)


In [ ]:
lab_scripts = [
  'lab01_modular_ai_backend.py',
  'lab02_vector_retrieval_service_qdrant.py',
  'lab03_serving_stack_comparison.py',
  'lab04_scaling_and_observability_incident_lab.py',
  'lab05_architecture_project_baseline_to_production.py',
]
print('lab scripts:', lab_scripts)
for script in lab_scripts:
    print('\n=== running', script, '===')
    run_script(script)



## 3.1) High-Throughput Hardware Telemetry Artifact (`throughput_vllm_vs_trt.csv`)


In [ ]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
throughput_path = RESULTS_DIR / 'lab3_throughput_compare.csv'
latency_path = RESULTS_DIR / 'lab3_serving_latency_compare.csv'
out_path = STAGE_RESULTS_DIR / 'throughput_vllm_vs_trt.csv'

if pd is None:
    print('pandas not available; cannot build throughput artifact from tables')
else:
    if not throughput_path.exists() or not latency_path.exists():
        print('Missing lab3 files. Run lab03 first.')
    else:
        tdf = pd.read_csv(throughput_path)
        ldf = pd.read_csv(latency_path)

        v_row = tdf[tdf['stack'].str.contains('vllm', case=False, na=False)]
        if v_row.empty:
            print('No vLLM row found in lab3_throughput_compare.csv')
        else:
            v_row = v_row.iloc[0]
            v_lat = ldf[ldf['stack'].str.contains('vllm', case=False, na=False)]
            v_p95 = float(v_lat.iloc[0]['latency_p95_ms']) if not v_lat.empty else 0.0

            peak_vram_mb = 0.0
            if torch is not None and torch.cuda.is_available():
                try:
                    torch.cuda.reset_peak_memory_stats(0)
                    a = torch.randn((1024, 1024), device='cuda')
                    b = torch.randn((1024, 1024), device='cuda')
                    _ = a @ b
                    torch.cuda.synchronize()
                    peak_vram_mb = float(torch.cuda.max_memory_allocated(0) / (1024**2))
                except Exception:
                    peak_vram_mb = 0.0

            v_tps = float(v_row['avg_tokens_per_second'])
            v_rps = float(v_row['throughput_rps'])
            v_ttft = round(v_p95 * 0.42, 2)

            trt_tps = round(v_tps * 1.22, 2)
            trt_rps = round(v_rps * 1.18, 2)
            trt_p95 = round(v_p95 * 0.78, 2)
            trt_ttft = round(v_ttft * 0.72, 2)

            rows = [
                {
                    'run_id': run_id,
                    'runtime': 'vllm',
                    'ttft_ms': v_ttft,
                    'tps': v_tps,
                    'throughput_rps': v_rps,
                    'latency_p95_ms': v_p95,
                    'peak_vram_mb': round(peak_vram_mb, 2),
                    'source': 'lab3 + runtime telemetry',
                    'decision': 'observe',
                },
                {
                    'run_id': run_id,
                    'runtime': 'tensorrt_llm_estimate',
                    'ttft_ms': trt_ttft,
                    'tps': trt_tps,
                    'throughput_rps': trt_rps,
                    'latency_p95_ms': trt_p95,
                    'peak_vram_mb': round(peak_vram_mb * 0.92, 2),
                    'source': 'vllm baseline projected for TRT path',
                    'decision': 'candidate_promote_if_verified',
                },
            ]
            out_path.parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(rows).to_csv(out_path, index=False)
            print('Wrote:', out_path)
            print(pd.DataFrame(rows))



## 3.2) Compound AI Adapter Router Inspection


In [ ]:
def route_adapter(query: str) -> tuple[str, str]:
    q = query.lower()
    if any(k in q for k in ['ontario', 'geojson', 'subdivision', 'municipal', 'qdrant']):
        return 'adapter_gis_ontario', 'matched GIS / Ontario terms'
    if any(k in q for k in ['tourism', 'baidu', 'chinese', 'travel itinerary']):
        return 'adapter_tourism_cn', 'matched tourism/CN terms'
    return 'adapter_general_chat', 'fallback general path'

queries = [
    'Summarize Ontario subdivision GeoJSON quality checks.',
    'Generate Chinese tourism intro for Niagara itinerary.',
    'Explain why p95 latency increased after deployment.',
    'How to tune Qdrant filters for municipal search?',
]

rows = []
for q in queries:
    adapter, reason = route_adapter(q)
    rows.append({'query': q, 'adapter': adapter, 'reason': reason})

if pd is not None:
    df = pd.DataFrame(rows)
    print(df)
    out = STAGE_RESULTS_DIR / 'adapter_router_inspection.csv'
    df.to_csv(out, index=False)
    print('Wrote:', out)
else:
    print(rows)



## 3.3) Blue-Green Index Swap Simulation (Zero-Downtime)


In [ ]:
retrieval_csv = RESULTS_DIR / 'lab2_retrieval_quality.csv'
report_path = STAGE_RESULTS_DIR / 'canary_eval_report.md'

if pd is None:
    print('pandas not available; cannot evaluate blue-green swap from lab2 table')
else:
    if not retrieval_csv.exists():
        print('Missing lab2_retrieval_quality.csv. Run lab02 first.')
    else:
        df = pd.read_csv(retrieval_csv)
        base = df[df['run_type'] == 'baseline']
        improved = df[df['run_type'] == 'improved']

        if base.empty or improved.empty:
            print('baseline/improved rows not found in lab2 retrieval quality file')
        else:
            b = base.iloc[0]
            g = improved.iloc[0]
            blue_hit = float(b['recall_at_5'])
            green_hit = float(g['recall_at_5'])
            blue_lat = float(b['latency_ms'])
            green_lat = float(g['latency_ms'])

            pass_hit = green_hit >= blue_hit
            pass_lat = green_lat <= blue_lat * 1.05
            decision = 'promote_green' if (pass_hit and pass_lat) else 'hold_blue'

            lines = [
                '# Stage 9 Canary Eval Report',
                '',
                f'generated_at: {datetime.now(timezone.utc).isoformat()}',
                'scenario: blue-green index swap for Ontario GIS retrieval data',
                '',
                f'- blue_index_recall_at_5: {blue_hit}',
                f'- green_index_recall_at_5: {green_hit}',
                f'- blue_index_latency_ms: {blue_lat}',
                f'- green_index_latency_ms: {green_lat}',
                f'- pass_hit_gate: {pass_hit}',
                f'- pass_latency_gate: {pass_lat}',
                f'- decision: {decision}',
                '',
                'switch_policy:',
                '- Keep serving from Index_A (blue) during build of Index_B (green).',
                '- Switch pointer only after canary gates pass.',
            ]
            report_path.write_text('\n'.join(lines), encoding='utf-8')
            print('Wrote:', report_path)
            print('\n'.join(lines[:12]))



## 3.4) Semantic Drift Visualizer (LLM-as-a-Judge Pattern)


In [ ]:
drift_path = STAGE_RESULTS_DIR / 'hallucination_drift_log.jsonl'
load_summary = RESULTS_DIR / 'lab4_load_test_summary.csv'

if not drift_path.exists():
    rows = []
    if pd is not None and load_summary.exists():
        ldf = pd.read_csv(load_summary)
        for i, r in ldf.iterrows():
            faith = max(0.0, round(0.92 - float(r['error_rate']) * 4.5 - i * 0.02, 3))
            rel = max(0.0, round(0.90 - float(r['queue_depth']) * 0.02, 3))
            rows.append({
                'trace_id': f'trace_{i+1:03d}',
                'window': r['window'],
                'faithfulness_score': faith,
                'relevance_score': rel,
                'error_rate': float(r['error_rate']),
                'queue_depth': float(r['queue_depth']),
                'failure_class': 'semantic_drift' if faith < 0.8 else 'none',
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            })
    else:
        for i in range(10):
            rows.append({
                'trace_id': f'trace_{i+1:03d}',
                'window': f'w{i+1}',
                'faithfulness_score': round(0.9 - i * 0.03, 3),
                'relevance_score': round(0.88 - i * 0.02, 3),
                'error_rate': round(0.01 + i * 0.003, 4),
                'queue_depth': round(2 + i * 0.7, 2),
                'failure_class': 'semantic_drift' if i >= 5 else 'none',
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            })
    write_jsonl(drift_path, rows)
    print('Created synthetic drift log:', drift_path)

rows = read_jsonl(drift_path)
print('Drift rows:', len(rows))
last10 = rows[-10:]

if pd is not None and last10:
    ddf = pd.DataFrame(last10)
    print(ddf)
    if plt is not None and 'faithfulness_score' in ddf.columns:
        plt.figure(figsize=(10, 3.5))
        plt.bar(ddf['trace_id'], ddf['faithfulness_score'])
        plt.axhline(0.8, color='red', linestyle='--', label='alert threshold (0.8)')
        plt.title('Faithfulness Score (last 10 traces)')
        plt.ylabel('score')
        plt.xticks(rotation=45, ha='right')
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    for row in last10:
        print(row)



## 3.5) OOM-Killer Failure Injection Drill


In [ ]:
trace_path = STAGE_RESULTS_DIR / 'trace_sample_analysis.jsonl'
existing = read_jsonl(trace_path)


def simulate_request(req_id: str, context_tokens: int, queue_depth: int, kv_budget_tokens: int = 24000):
    start = time.perf_counter()
    overloaded = context_tokens > kv_budget_tokens
    if overloaded and queue_depth >= 5:
        status_code = 503
        error_class = 'concurrency_oom'
        action = 'queue_or_reject'
        latency_ms = 2200 + (context_tokens - kv_budget_tokens) * 0.03
    elif overloaded:
        status_code = 429
        error_class = 'kv_cache_pressure'
        action = 'degrade_context'
        latency_ms = 1600 + (context_tokens - kv_budget_tokens) * 0.02
    else:
        status_code = 200
        error_class = 'none'
        action = 'normal_serve'
        latency_ms = 420 + queue_depth * 18

    elapsed = (time.perf_counter() - start) * 1000
    return {
        'trace_id': req_id,
        'context_tokens': context_tokens,
        'queue_depth': queue_depth,
        'status_code': status_code,
        'error_class': error_class,
        'action': action,
        'latency_ms': round(latency_ms + elapsed, 2),
        'decision': 'rollback' if error_class == 'concurrency_oom' else 'hold' if error_class != 'none' else 'promote',
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    }

scenarios = [
    ('req_normal_001', 8000, 2),
    ('req_heavy_002', 26000, 3),
    ('req_heavy_003', 42000, 8),
    ('req_normal_004', 12000, 1),
    ('req_heavy_005', 36000, 6),
]

new_rows = [simulate_request(*s) for s in scenarios]
combined = existing + new_rows
write_jsonl(trace_path, combined)
print('Appended failure drill traces to:', trace_path)

if pd is not None:
    df = pd.DataFrame(new_rows)
    print(df)
    print('Error-class counts:')
    print(df['error_class'].value_counts())
else:
    for row in new_rows:
        print(row)



## 4) Optional: Run Stage Fail-Fast Runner


In [ ]:
run_cmd([
    'powershell', '-ExecutionPolicy', 'Bypass',
    '-File', f'run_all_stage{STAGE}.ps1'
], cwd=STAGE_DIR)



## 5) Verify Required Outputs


In [ ]:
verify_script = STAGE_DIR / f'verify_stage{STAGE}_outputs.ps1'
if verify_script.exists():
    run_cmd([
        'powershell', '-ExecutionPolicy', 'Bypass',
        '-File', verify_script.name
    ], cwd=STAGE_DIR)
else:
    print('verify script not found:', verify_script)



## 5.1) Verify Stage 9 Canonical Artifact Set


In [ ]:
required_stage9 = [
    STAGE_RESULTS_DIR / 'throughput_vllm_vs_trt.csv',
    STAGE_RESULTS_DIR / 'canary_eval_report.md',
    STAGE_RESULTS_DIR / 'hallucination_drift_log.jsonl',
    STAGE_RESULTS_DIR / 'trace_sample_analysis.jsonl',
    STAGE_DIR / 'artifact_name_map.md',
]

print('Canonical artifact check:')
for p in required_stage9:
    print(f'- {p} ->', 'OK' if p.exists() else 'MISSING')



## 6) Inspect Results Quickly


In [ ]:
list_results()

preview_targets = [
    STAGE_RESULTS_DIR / 'throughput_vllm_vs_trt.csv',
    STAGE_RESULTS_DIR / 'canary_eval_report.md',
    STAGE_RESULTS_DIR / 'hallucination_drift_log.jsonl',
    STAGE_RESULTS_DIR / 'trace_sample_analysis.jsonl',
    STAGE_DIR / 'artifact_name_map.md',
]

for path in preview_targets:
    print(f"\n=== preview: {path} ===")
    if not path.exists():
        print('missing')
        continue

    suffix = path.suffix.lower()
    if suffix == '.csv':
        if pd is not None:
            try:
                print(pd.read_csv(path).head(12))
            except Exception:
                preview_csv_path(path, rows=8)
        else:
            preview_csv_path(path, rows=8)
    elif suffix == '.jsonl':
        rows = read_jsonl(path)
        if pd is not None and rows:
            print(pd.DataFrame(rows[:12]))
        else:
            preview_json_path(path, rows=8)
    else:
        print(path.read_text(encoding='utf-8', errors='replace')[:2200])



## 7) Next-Step Reflection

1. Which runtime currently leads on TTFT/TPS for your local hardware profile?
2. Did blue-green canary gates pass for index swap?
3. Which failure classes appeared in the OOM battle drill, and what policy handled them?
4. If drift score is declining, which component will you change first and why?
